In [1]:
import pandas as pd
import numpy as np

mathys_expr_path = "/n/groups/patel/adithya/CellMatrix_with_genenames.parquet"

# Just peek at a slice — don't load the full thing twice
m_expr = pd.read_parquet(mathys_expr_path)
m_expr.set_index('index', inplace=True)
print("Mathys matrix shape (genes x cells, pre-transpose):", m_expr.shape)
print("dtype:", m_expr.dtypes.iloc[0])

# Sample a 200-cell x all-genes block to check value distribution
sample = m_expr.iloc[:, :200].T  # cells x genes
print("\nValue stats on 200-cell sample:")
print(f"  min            = {sample.values.min():.4f}")
print(f"  max            = {sample.values.max():.4f}")
print(f"  mean nonzero   = {sample.values[sample.values > 0].mean():.4f}")
print(f"  % zeros        = {(sample.values == 0).mean() * 100:.1f}")
print(f"  per-cell sums (first 5): {sample.sum(axis=1).iloc[:5].values}")
print(f"  integer-valued?: {np.allclose(sample.values, sample.values.astype(int))}")

Mathys matrix shape (genes x cells, pre-transpose): (17926, 70634)
dtype: int64

Value stats on 200-cell sample:
  min            = 0.0000
  max            = 53.0000
  mean nonzero   = 2.0405
  % zeros        = 82.3
  per-cell sums (first 5): [ 296 8328  314 8017 4410]
  integer-valued?: True


In [2]:
sa_meta_path = "/n/groups/patel/adithya/SEAAD_Outputs/SEAAD_CellMetadata.parquet"
sa_meta = pd.read_parquet(sa_meta_path)
print("SEA-AD metadata shape:", sa_meta.shape)
print("\nColumns:", list(sa_meta.columns))
print("\nbroad_cell_type counts:")
print(sa_meta['broad_cell_type'].value_counts())
print("\nalzheimers_or_control counts:")
print(sa_meta['alzheimers_or_control'].value_counts())
print("\nN donors:", sa_meta['Donor ID'].nunique())
print("\nFirst row:")
print(sa_meta.head(1).T)

SEA-AD metadata shape: (1297754, 11)

Columns: ['Donor ID', 'Sex', 'Age at Death', 'PMI', 'APOE Genotype', 'Cognitive Status', 'Overall AD neuropathological Change', 'Subclass', 'Neurotypical reference', 'broad_cell_type', 'alzheimers_or_control']

broad_cell_type counts:
broad_cell_type
Ex     610593
Inh    393273
Oli    142064
Ast     83983
Mic     41187
Opc     26654
Name: count, dtype: int64

alzheimers_or_control counts:
alzheimers_or_control
0    702646
1    595108
Name: count, dtype: int64

N donors: 80

First row:
TAG                                 CCAAGCGAGCTTCATG-L8TX_210624_01_C03-1114947893
Donor ID                                                                H20.33.029
Sex                                                                         Female
Age at Death                                                                  91.0
PMI                                                                            6.1
APOE Genotype                                            

In [3]:
sa_ex_path = "/n/groups/patel/adithya/SEAAD_Outputs/SEAAD_Matrix_Ex.parquet"
sa_ex = pd.read_parquet(sa_ex_path)
print("SEA-AD Ex matrix shape (cells x genes):", sa_ex.shape)
print("dtype:", sa_ex.dtypes.iloc[0])
print("First 3 gene names:", list(sa_ex.columns[:3]))
print("Index name / sample:", sa_ex.index.name, "/", list(sa_ex.index[:3]))

sample = sa_ex.iloc[:200, :]  # 200 cells x all genes (cells already on rows here)
print("\nValue stats on 200-cell sample:")
print(f"  min            = {sample.values.min():.4f}")
print(f"  max            = {sample.values.max():.4f}")
print(f"  mean nonzero   = {sample.values[sample.values > 0].mean():.4f}")
print(f"  % zeros        = {(sample.values == 0).mean() * 100:.1f}")
print(f"  per-cell sums (first 5): {sample.sum(axis=1).iloc[:5].values}")
print(f"  integer-valued?: {np.allclose(sample.values, sample.values.astype(int))}")

SEA-AD Ex matrix shape (cells x genes): (610593, 36601)
dtype: float32
First 3 gene names: ['MIR1302-2HG', 'FAM138A', 'OR4F5']
Index name / sample: TAG / ['CCCTAACAGTCGAAGC-L8TX_210527_01_D07-1108004948', 'AAACGCTGTTCGCGTG-L8TX_220217_01_H04-1162250581', 'CCCTCTCTCCTGTAGA-L8TX_210930_01_H02-1134006386']

Value stats on 200-cell sample:
  min            = 0.0000
  max            = 16.4710
  mean nonzero   = 6.0004
  % zeros        = 79.8
  per-cell sums (first 5): [41262.97  28565.55  48435.305 42244.457 46983.984]
  integer-valued?: False


In [4]:
markers_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_pseudotime/mathys_markers.xlsx"
mk = pd.read_excel(markers_path, sheet_name="Subpopulation_markers", header=1)
mk = mk[['gene.name', 'adj.pvals', 'lFC', 'subpopulation']].dropna(subset=['subpopulation'])
print("Total marker rows:", len(mk))
print("Subpopulations:", sorted(mk['subpopulation'].unique().tolist()))

# Per-subpop gene list dict (will use these as gene sets for sc.tl.score_genes later)
mathys_markers = {
    sp: mk.loc[mk['subpopulation'] == sp, 'gene.name'].tolist()
    for sp in sorted(mk['subpopulation'].unique())
}

# Overlap with SEA-AD Ex genes (assumes Cell 3 ran; reuses sa_ex.columns)
sa_genes = set(sa_ex.columns)
print("\nMarker overlap with SEA-AD Ex gene panel:")
print(f"{'subpop':>6} | {'n_markers':>9} | {'n_overlap':>9} | {'% overlap':>9}")
for sp, genes in mathys_markers.items():
    n = len(genes)
    nov = sum(g in sa_genes for g in genes)
    print(f"{sp:>6} | {n:>9} | {nov:>9} | {100*nov/max(n,1):>8.1f}%")

Total marker rows: 9364
Subpopulations: ['Ast0', 'Ast1', 'Ast2', 'Ast3', 'Ex0', 'Ex1', 'Ex11', 'Ex12', 'Ex14', 'Ex2', 'Ex3', 'Ex4', 'Ex5', 'Ex6', 'Ex7', 'Ex8', 'Ex9', 'In0', 'In1', 'In10', 'In11', 'In2', 'In3', 'In4', 'In5', 'In6', 'In7', 'In8', 'In9', 'Mic0', 'Mic1', 'Mic2', 'Mic3', 'Oli0', 'Oli1', 'Oli3', 'Oli4', 'Oli5', 'Opc0', 'Opc1', 'Opc2']

Marker overlap with SEA-AD Ex gene panel:
subpop | n_markers | n_overlap | % overlap
  Ast0 |        33 |        33 |    100.0%
  Ast1 |        72 |        70 |     97.2%
  Ast2 |        71 |        71 |    100.0%
  Ast3 |       232 |       228 |     98.3%
   Ex0 |       345 |       327 |     94.8%
   Ex1 |       302 |       294 |     97.4%
  Ex11 |       772 |       740 |     95.9%
  Ex12 |       362 |       353 |     97.5%
  Ex14 |       538 |       524 |     97.4%
   Ex2 |        52 |        51 |     98.1%
   Ex3 |       207 |       206 |     99.5%
   Ex4 |       231 |       223 |     96.5%
   Ex5 |       217 |       213 |     98.2%
   Ex6

In [5]:
# A. What ARE Ex2's 52 markers? Are they pan-Ex (SLC17A7, RBFOX3, NEUROD2, NEUROD6, SATB2, etc.)
#    or Ex2-specific?
import pandas as pd
mk = pd.read_excel(
    '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/seaad_pseudotime/mathys_markers.xlsx',
    sheet_name='Subpopulation_markers', header=1
)
mk = mk[['gene.name', 'adj.pvals', 'lFC', 'subpopulation']].dropna(subset=['subpopulation'])
ex_subpops = sorted(mk.loc[mk['subpopulation'].astype(str).str.match(r'^Ex\d+$'),
                           'subpopulation'].unique())

# Ex2 markers, sorted by lFC
ex2 = mk[mk['subpopulation'] == 'Ex2'].sort_values('lFC', ascending=False)
print("Ex2 markers (52 total), top 20 by lFC:")
print(ex2.head(20)[['gene.name', 'lFC']].to_string(index=False))

# B. How many of Ex2's markers also appear in OTHER Ex subpops' marker lists?
ex2_genes = set(ex2['gene.name'])
print("\nEx2 marker overlap with each other Ex subpop:")
for sp in ex_subpops:
    if sp == 'Ex2':
        continue
    other_genes = set(mk.loc[mk['subpopulation'] == sp, 'gene.name'])
    shared = ex2_genes & other_genes
    print(f"  Ex2 ∩ {sp}: {len(shared)}/52 ({100*len(shared)/52:.0f}%) — {sorted(shared)[:5]}")

Ex2 markers (52 total), top 20 by lFC:
  gene.name      lFC
    SLC26A3 1.649196
       DHFR 1.633113
   RASGEF1B 1.540430
      CBLN2 1.059114
     KCNIP4 1.039740
     LINGO1 0.995414
      KCND2 0.950163
      CSMD1 0.874038
      EPHA6 0.818762
     ARPP21 0.818452
      GRIA4 0.816800
      GRIN1 0.809894
     LRRTM4 0.795739
    STXBP5L 0.790238
      TENM2 0.777161
    CNTNAP5 0.765893
IQCJ-SCHIP1 0.743208
       SYN3 0.697176
      FLRT2 0.686646
     PHYHIP 0.679884

Ex2 marker overlap with each other Ex subpop:
  Ex2 ∩ Ex0: 13/52 (25%) — ['CBLN2', 'CDH12', 'CNTNAP5', 'DGKB', 'EPHA6']
  Ex2 ∩ Ex1: 3/52 (6%) — ['CNTN5', 'PCDH7', 'RORA']
  Ex2 ∩ Ex11: 1/52 (2%) — ['CDH12']
  Ex2 ∩ Ex12: 4/52 (8%) — ['CNTNAP5', 'NEGR1', 'PTPRD', 'SLC8A1']
  Ex2 ∩ Ex14: 3/52 (6%) — ['DHFR', 'RORA', 'SNTG1']
  Ex2 ∩ Ex3: 1/52 (2%) — ['CDH12']
  Ex2 ∩ Ex4: 9/52 (17%) — ['DGKB', 'DHFR', 'EPHA6', 'FLRT2', 'GRIN1']
  Ex2 ∩ Ex5: 5/52 (10%) — ['ADGRL2', 'CNTN5', 'FAM19A2', 'RORA', 'SNTG1']
  Ex2 ∩ Ex6: 0